# 15.11 少算一些就一定更快嗎？

# 第 15 章：從 Dense 到 Mixture of Experts（MoE）

前置：第4章Dense FFN。把token想成訂單，experts像不同廚師，router像分單員。這個譬喻只描述分工：專家不會天生知道中文或數學，專長需由資料與訓練觀察。實作是dropless小型MoE，不需分散式系統。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：四位師傅都在，這張訂單只請兩位動手**

例中只派給A與C；未選中師傅的權重仍佔儲存空間。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/moe.svg")))

**先想一想：**四位小廚師加分單，會一定快過一位大廚師嗎？

少做理論乘法不保證更快。分單、索引、小矩陣與Python loop都有成本；小CPU模型MoE可能更慢。

**把直覺寫成數學：**wall time需warmup與重複，GPU另同步；total/active參數不是速度測量。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
import time
from tiny_perceptron.modern import DenseFFN, MoEFFN

x = torch.randn(2, 8, 8)
for name, layer in (("Dense", DenseFFN(8)), ("MoE", MoEFFN(8, experts=4, top_k=2))):
    layer(x)
    start = time.perf_counter()
    for _ in range(10):
        layer(x)
    print(name, (time.perf_counter() - start) / 10, "秒/forward，本機tiny測量")

**如何讀結果：**當前函數沒有優化dispatch kernel，不能把CPU結果外推到大GPU多節點。

**只改一件事：**只增加token數T，觀察相對開銷。
